In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/df.csv')

#Training-Test Split (Anti Data Leakage Barrier)

##Popping out the Target and spliting the data.

In [18]:
target_name = 'survived'

In [19]:
X = df.drop(columns = [target_name])

Y = df['survived']

In [20]:
# Create the function to split the dataset

def create_train_val_test(X, Y, seed = 22):

  #Create the training set 70%, validation 15% and testing 15%

  xtr, x_rest, ytr, y_rest = train_test_split(X, Y, test_size = 0.3, random_state= seed, stratify= Y)

  xvl, xts, yvl, yts = train_test_split(x_rest, y_rest, test_size = 0.5, random_state = seed, stratify = y_rest)

  return xtr, ytr, xvl, yvl, xts, yts

# Execute the function
x_train, y_train, x_val, y_val, x_test, y_test = create_train_val_test(X, Y)

# Verify
print(f'Dimentions of x & y train: {x_train.shape, y_train.shape}')
print(f'Dimentions of x & y val: {x_val.shape, y_val.shape}')
print(f'Dimentions of x & y test: {x_test.shape, y_test.shape}')

Dimentions of x & y train: ((623000, 14), (623000,))
Dimentions of x & y val: ((133500, 14), (133500,))
Dimentions of x & y test: ((133500, 14), (133500,))


##Getting X and Y together for the EDA

In [21]:

df_eda = x_train.copy()

df_eda[target_name] = y_train


print(f'x_train {type(x_train)}')
print(f'y_train {type(y_train)}')

df_eda.head(5)

x_train <class 'pandas.core.frame.DataFrame'>
y_train <class 'pandas.core.series.Series'>


,age,gender,diagnosis_date,cancer_stage,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,survived
839821,35.0,Male,2022-12-08,Stage III,No,Passive Smoker,24.5,162,1,0,1,0,Chemotherapy,2023-08-29,0
315360,59.0,Male,2019-09-28,Stage IV,No,Former Smoker,35.6,265,0,0,0,0,Surgery,2021-03-16,0
555070,33.0,Male,2016-06-15,Stage IV,No,Current Smoker,35.9,242,1,0,0,0,Chemotherapy,2017-03-26,0
224728,59.0,Male,2023-01-20,Stage I,No,Former Smoker,38.1,298,1,1,0,0,Surgery,2024-04-01,1
349925,50.0,Male,2020-01-06,Stage IV,Yes,Current Smoker,25.7,231,1,0,0,1,Chemotherapy,2021-05-31,0


In [22]:
df_eda.to_csv('/content/drive/MyDrive/Colab Notebooks/df_eda.csv', inded = False)

Data splitted for the training, validation and testing sets, preventing data leakeage.

Irrelevant columns dropped and EDA data set saved in Google Drive with the Training data and the target columns for deep analysis.

-----------------------------------------------------------------------------------------------------------------------------------

#Preprocessing & Feature Engineering

##Ordinal Mapping

MAPPING ''CANCER STAGE'' COLUMN

In [23]:
df['cancer_stage'].unique()

array(['Stage I', 'Stage III', 'Stage IV', 'Stage II'], dtype=object)

In [24]:
def cancer_mapping(data):

  stage_mapping = {'Stage I': 1, 'Stage II': 2, 'Stage III': 3, 'Stage IV': 4}
  data['cancer_stage_mapped'] = data['cancer_stage'].map(stage_mapping)

  data.drop(columns = ['cancer_stage'], errors = 'ignore', inplace = True)

  return data

x_test = cancer_mapping(x_test)
x_train = cancer_mapping(x_train)
x_val = cancer_mapping(x_val)

x_train.head()

,age,gender,diagnosis_date,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,cancer_stage_mapped
839821,35.0,Male,2022-12-08,No,Passive Smoker,24.5,162,1,0,1,0,Chemotherapy,2023-08-29,3
315360,59.0,Male,2019-09-28,No,Former Smoker,35.6,265,0,0,0,0,Surgery,2021-03-16,4
555070,33.0,Male,2016-06-15,No,Current Smoker,35.9,242,1,0,0,0,Chemotherapy,2017-03-26,4
224728,59.0,Male,2023-01-20,No,Former Smoker,38.1,298,1,1,0,0,Surgery,2024-04-01,1
349925,50.0,Male,2020-01-06,Yes,Current Smoker,25.7,231,1,0,0,1,Chemotherapy,2021-05-31,4


MAPPING ''SMOKING STATUS'' COLUMN

In [25]:
df['smoking_status'].unique()

array(['Passive Smoker', 'Former Smoker', 'Never Smoked',
       'Current Smoker'], dtype=object)

In [26]:
def smoking_status_mapper(data):

  smoking_status_mapped = {"Never Smoked": 1, 'Passive Smoker' : 2, "Former Smoker" : 3, "Current Smoker" : 4}
  data['smoking_status_risk'] = data['smoking_status'].map(smoking_status_mapped)

  data.drop(columns = ['smoking_status'], errors = 'ignore', inplace = True)

  return data

x_train = smoking_status_mapper(x_train)
x_val = smoking_status_mapper(x_val)
x_test = smoking_status_mapper(x_test)

x_train.head()

,age,gender,diagnosis_date,family_history,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,cancer_stage_mapped,smoking_status_risk
839821,35.0,Male,2022-12-08,No,24.5,162,1,0,1,0,Chemotherapy,2023-08-29,3,2
315360,59.0,Male,2019-09-28,No,35.6,265,0,0,0,0,Surgery,2021-03-16,4,3
555070,33.0,Male,2016-06-15,No,35.9,242,1,0,0,0,Chemotherapy,2017-03-26,4,4
224728,59.0,Male,2023-01-20,No,38.1,298,1,1,0,0,Surgery,2024-04-01,1,3
349925,50.0,Male,2020-01-06,Yes,25.7,231,1,0,0,1,Chemotherapy,2021-05-31,4,4


Converting treatment Dates columns (High Cardinality columns) to full period of treatment single-column then dropping them before ONE-HOT Encoding.

In [27]:
def converter_treatment_dates(data):

  data['diagnosis_date'] = pd.to_datetime(data['diagnosis_date'])
  data['end_treatment_date'] = pd.to_datetime(data['end_treatment_date'])

  data['treatment_duration_days'] = (data['end_treatment_date'] - data['diagnosis_date']).dt.days

  data.drop(columns = ['diagnosis_date', 'end_treatment_date'], inplace = True)

  return data

x_train = converter_treatment_dates(x_train)
x_val = converter_treatment_dates(x_val)
x_test = converter_treatment_dates(x_test)

x_train.head()

,age,gender,family_history,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,cancer_stage_mapped,smoking_status_risk,treatment_duration_days
839821,35.0,Male,No,24.5,162,1,0,1,0,Chemotherapy,3,2,264
315360,59.0,Male,No,35.6,265,0,0,0,0,Surgery,4,3,535
555070,33.0,Male,No,35.9,242,1,0,0,0,Chemotherapy,4,4,284
224728,59.0,Male,No,38.1,298,1,1,0,0,Surgery,1,3,437
349925,50.0,Male,Yes,25.7,231,1,0,0,1,Chemotherapy,4,4,511


##One-Hot Encoding

In [30]:
def ohe_and_align(x_train, x_val, x_test):

  categorical_cols = x_train.select_dtypes(include = ['object', 'category']).columns.tolist()

  if len(categorical_cols) > 0:

    x_val = pd.get_dummies(x_val, columns = categorical_cols, drop_first= True, dtype=int)
    x_test = pd.get_dummies(x_test, columns = categorical_cols, drop_first= True, dtype=int)
    x_train = pd.get_dummies(x_train, columns = categorical_cols, drop_first= True, dtype=int)

    val_aligned = x_val.reindex(columns = x_train.columns, fill_value= 0)
    test_aligned = x_test.reindex(columns = x_train.columns, fill_value= 0)

  return x_train, val_aligned, test_aligned

x_train, x_val, x_test = ohe_and_align(x_train, x_val, x_test)

x_train.head()

,age,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,cancer_stage_mapped,smoking_status_risk,treatment_duration_days,gender_Male,family_history_Yes,treatment_type_Combined,treatment_type_Radiation,treatment_type_Surgery
839821,35.0,24.5,162,1,0,1,0,3,2,264,1,0,0,0,0
315360,59.0,35.6,265,0,0,0,0,4,3,535,1,0,0,0,1
555070,33.0,35.9,242,1,0,0,0,4,4,284,1,0,0,0,0
224728,59.0,38.1,298,1,1,0,0,1,3,437,1,0,0,0,1
349925,50.0,25.7,231,1,0,0,1,4,4,511,1,1,0,0,0


##Feature Scaling (StandardScaler)

In [31]:
def scale_numerical_features(train_data, val_data, test_data):

  columns_to_scale = ['age', 'bmi', 'cholesterol_level', 'treatment_duration_days']

  scaler = StandardScaler()

  train_data[columns_to_scale] = scaler.fit_transform(train_data[columns_to_scale])

  val_data[columns_to_scale] = scaler.transform(val_data[columns_to_scale])
  test_data[columns_to_scale] = scaler.transform(test_data[columns_to_scale])

  return train_data, val_data, test_data

x_train, x_val, x_test = scale_numerical_features(x_train, x_val, x_test)

x_train.head()

,age,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,cancer_stage_mapped,smoking_status_risk,treatment_duration_days,gender_Male,family_history_Yes,treatment_type_Combined,treatment_type_Radiation,treatment_type_Surgery
839821,-2.003309,-0.716383,-1.648215,1,0,1,0,3,2,-1.392110,1,0,0,0,0
315360,0.398426,0.610744,0.722538,0,0,0,0,4,3,0.552434,1,0,0,0,1
555070,-2.203453,0.646612,0.193147,1,0,0,0,4,4,-1.248601,1,0,0,0,0
224728,0.398426,0.909646,1.482100,1,1,0,0,1,3,-0.150759,1,0,0,0,1
349925,-0.502225,-0.572910,-0.060040,1,0,0,1,4,4,0.380223,1,1,0,0,0


In [34]:
x_train.to_csv('/content/drive/MyDrive/Colab Notebooks/x_train.csv', index = False)
x_val.to_csv('/content/drive/MyDrive/Colab Notebooks/x_val.csv', index = False)
x_test.to_csv('/content/drive/MyDrive/Colab Notebooks/x_test.csv', index = False)
y_train.to_csv('/content/drive/MyDrive/Colab Notebooks/y_train.csv', index = False)
y_val.to_csv('/content/drive/MyDrive/Colab Notebooks/y_val.csv', index = False)
y_test.to_csv('/content/drive/MyDrive/Colab Notebooks/y_test.csv', index = False)


We finish this phase of the project processing the categorical data and mapping columns, cleaning and seting up the data for the modeling phase.